In [1]:
from mc_experiment import (
    make_seed_counter,
    next_seed,
    standardize_innovations,
    summarize_reference_experiment,
    summarize_mle_augmentation_experiment,
    augmented_config_path,
)

from SymbolicDSGE import ModelParser, DSGESolver, Shock
from SymbolicDSGE.bayesian import make_prior

from numpy import log
import numpy as np

from scipy.stats import chi2, gaussian_kde, norm

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl

from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
import cProfile

import contextlib
import io
STDOUT_VOID = lambda: contextlib.redirect_stdout(io.StringIO())

_KNOWN_R = True
_AUGMENTED_PARAM = 'x_coef'
_AUGMENTED_EQUATION = 'Rate'
_AUGMENTED_CONFIG = augmented_config_path(_AUGMENTED_EQUATION)
_MEAS_ERR_SCALE = 0.25
_MC_SAMPLES = 100_000
_MC_ALPHA = 0.05
_MC_SUMMARY_ONLY = True
_MC_INCLUDE_BY_PREDICTOR = False
_FIGSIZE_1D = (10, 6)
_FIGSIZE_2D = (12, 6)


Detected IPython. Loading juliacall extension. See https://juliapy.github.io/PythonCall.jl/stable/compat/#IPython


In [2]:
# Load reference model
parser = ModelParser("../../MODELS/misspec_test/reference.yaml")
config, kalman = parser.get_all()
solver = DSGESolver(config, kalman)

comp = solver.compile(
    n_state=3,
    n_exog=3,
)
sol = solver.solve(
    comp,
    steady_state=[0.0, 0.0, 0.0, 0.0, 0.0],
)

print("Transition matrix:\n", sol.A.round(3), "\n")
print("Shock Loadings:\n", sol.B.round(3))

Transition matrix:
 [[ 0.83  -0.     0.     0.     0.   ]
 [ 0.     0.85   0.     0.     0.   ]
 [ 0.288 -0.047  0.28   0.     0.   ]
 [ 0.892  0.708 -1.711  0.     0.   ]
 [ 0.7   -0.115 -1.363  0.     0.   ]] 

Shock Loadings:
 [[ 1.     0.     0.   ]
 [ 0.     1.     0.   ]
 [ 0.     0.     1.   ]
 [ 3.193  0.493 -6.107]
 [ 2.531 -0.406 -4.864]]


In [3]:
# Load Misspecified DGP
parser_dgp = ModelParser("../../MODELS/misspec_test/misspec.yaml")
config_dgp, kalman_dgp = parser_dgp.get_all()
solver_dgp = DSGESolver(config_dgp, kalman_dgp)
comp_dgp = solver_dgp.compile(
    n_state=3,
    n_exog=3,
)
sol_dgp = solver_dgp.solve(
    comp_dgp,
    steady_state=[0.0, 0.0, 0.0, 0.0, 0.0],
)

In [4]:
# Large sample simulations used to approximate measurement-noise variances
_large_sample_seed_counter = make_seed_counter(start=100_000)
shocks_large = {
    "g,z": Shock(10_000, "norm", multivar=True, seed=next_seed(_large_sample_seed_counter)).shock_generator(),
    "r": Shock(10_000, "norm", multivar=False, seed=next_seed(_large_sample_seed_counter)).shock_generator(),
}

sim1 = sol_dgp.sim(
    T=10_000,
    shocks=shocks_large,
    observables=True,
)

sim2 = sol.sim(
    T=10_000,
    shocks=shocks_large,
    observables=True,
)

In [5]:
T = 200
_plot_seed_counter = make_seed_counter(start=2_000_000)

err_var = np.var(np.column_stack([sim1["OutGap"], sim1["Infl"], sim1["Rate"]]), axis=0)
mc_reference = summarize_reference_experiment(
    sol,
    sol_dgp,
    T=T,
    err_var=err_var,
    meas_err_scale=_MEAS_ERR_SCALE,
    mc_samples=_MC_SAMPLES,
    known_r=_KNOWN_R,
    alpha=_MC_ALPHA,
    summary_only=_MC_SUMMARY_ONLY,
    include_by_predictor=_MC_INCLUDE_BY_PREDICTOR,
)

rep_ref = mc_reference["representative"]
sim_dgp = rep_ref.sim_dgp
obs = rep_ref.obs
kf = rep_ref.kf
std_innov = rep_ref.std_innov
err_scale = rep_ref.err_scale
N, n_obs = kf.innov.shape

_measurement_order = {"OutGap": 0, "Infl": 1, "Rate": 2}
_predictor_order = {"Pi": 0, "x": 1, "r": 2}

def _sort_summary(df):
    out = df.copy()
    if "measurement" in out.columns:
        out["measurement_order"] = out["measurement"].map(_measurement_order)
    if "predictor" in out.columns:
        out["predictor_order"] = out["predictor"].map(_predictor_order)
    if "target" in out.columns:
        out["target_order"] = out["target"].map(_predictor_order)
    if "regressor" in out.columns:
        out["regressor_order"] = out["regressor"].map(_predictor_order)
    sort_cols = [
        col
        for col in ["measurement_order", "target_order", "predictor_order", "regressor_order"]
        if col in out.columns
    ]
    if sort_cols:
        out = out.sort_values(sort_cols)
    return out.drop(columns=[col for col in ["measurement_order", "target_order", "predictor_order", "regressor_order"] if col in out.columns])

sim_ref = sol.sim(
    T=T,
    shocks={
        "g,z": Shock(T, "norm", multivar=True, seed=next_seed(_plot_seed_counter)).shock_generator(),
        "r": Shock(T, "norm", multivar=False, seed=next_seed(_plot_seed_counter)).shock_generator(),
    },
    observables=True,
)
ref = np.column_stack([sim_ref["OutGap"], sim_ref["Infl"], sim_ref["Rate"]])[1:, :]

obs_dgp = np.column_stack([sim1["OutGap"], sim1["Infl"], sim1["Rate"]])[1:, :]
if np.any(err_scale != 0.0):
    _plot_rng = np.random.default_rng(next_seed(_plot_seed_counter))
    obs_dgp = obs_dgp + _plot_rng.normal(scale=np.sqrt(err_scale), size=obs_dgp.shape)

In [6]:
print(f"Known R assumption: {_KNOWN_R}")
print(f"Augmented measurement equation: {_AUGMENTED_EQUATION}")
print(f"Augmented coefficient: {_AUGMENTED_PARAM}")
print(f"Monte Carlo replications: {_MC_SAMPLES}")
print("Noise Covariance:\n", np.diag(err_scale).round(3))


Known R assumption: True
Augmented measurement equation: Rate
Augmented coefficient: x_coef
Monte Carlo replications: 100000
Noise Covariance:
 [[3.021 0.    0.   ]
 [0.    4.196 0.   ]
 [0.    0.    0.195]]


In [7]:
print(f"Monte Carlo Ljung-Box summary across {_MC_SAMPLES} replications:")
display(mc_reference["lb_summary"].round(3))

Monte Carlo Ljung-Box summary across 100000 replications:


,measurement,lb_stat,p_value,mc_se_lb_stat,mc_se_p_value,n_replications,n_rejections,reject_rate,reject_rate_mc_se,reject_ci_low,reject_ci_high
0,OutGap,1.233,0.461,0.005,0.001,100000,7643,0.076,0.001,0.075,0.078
1,Infl,1.758,0.393,0.007,0.001,100000,14093,0.141,0.001,0.139,0.143
2,Rate,1.004,0.497,0.004,0.001,100000,5011,0.050,0.001,0.049,0.051


In [8]:
print(f"Moment Tests summary across {_MC_SAMPLES} replications:")
display(mc_reference["moment_specification_test_summary"].round(3))

Moment Tests summary across 100000 replications:


,test,distance,stat,p_value,mc_se_distance,mc_se_stat,mc_se_p_value,n_replications,n_rejections,reject_rate,reject_rate_mc_se,reject_ci_low,reject_ci_high,df,sample_size,bandwidth
0,mean_zero_hac,0.119,3.175,0.491,0.000,0.009,0.001,100000,6697,0.067,0.001,0.065,0.069,3.0,200,4
1,cov_identity,1.700,162.074,0.000,0.001,0.193,0.000,100000,100000,1.000,0.000,1.000,1.000,6.0,200,4


In [9]:
print("Innovations on orthogonalized predicted states (Monte Carlo averages and rejection rates):")
_sort_summary(mc_reference["measurement_regressions_orthogonalized_summary"]).round(3)

Innovations on orthogonalized predicted states (Monte Carlo averages and rejection rates):


,measurement,predictor,coef,standardized_coef,std_error,t_stat,p_value,r2,mc_se_coef,mc_se_standardized_coef,mc_se_std_error,mc_se_t_stat,mc_se_p_value,mc_se_r2,n_replications,n_rejections,reject_rate,reject_rate_mc_se,reject_ci_low,reject_ci_high
0,OutGap,Pi,0.982,0.052,1.321,0.744,0.417,0.008,0.004,0.0,0.000,0.003,0.001,0.0,100000,12020,0.120,0.001,0.118,0.122
1,OutGap,x,-0.386,-0.076,0.337,-1.081,0.349,0.010,0.001,0.0,0.000,0.003,0.001,0.0,100000,17708,0.177,0.001,0.175,0.179
2,OutGap,r,-1.752,-0.046,2.642,-0.647,0.435,0.007,0.009,0.0,0.001,0.003,0.001,0.0,100000,10150,0.102,0.001,0.100,0.103
3,Infl,Pi,-0.779,-0.034,1.574,-0.475,0.465,0.006,0.005,0.0,0.000,0.003,0.001,0.0,100000,7631,0.076,0.001,0.075,0.078
4,Infl,x,0.068,0.013,0.403,0.187,0.489,0.005,0.001,0.0,0.000,0.003,0.001,0.0,100000,5771,0.058,0.001,0.056,0.059
5,Infl,r,-0.619,-0.011,3.150,-0.157,0.493,0.005,0.010,0.0,0.001,0.003,0.001,0.0,100000,5504,0.055,0.001,0.054,0.056
6,Rate,Pi,-0.081,-0.019,0.315,-0.265,0.482,0.006,0.001,0.0,0.000,0.003,0.001,0.0,100000,6388,0.064,0.001,0.062,0.065
7,Rate,x,0.048,0.041,0.081,0.579,0.445,0.007,0.000,0.0,0.000,0.003,0.001,0.0,100000,9507,0.095,0.001,0.093,0.097
8,Rate,r,-0.136,-0.012,0.630,-0.174,0.486,0.005,0.002,0.0,0.000,0.003,0.001,0.0,100000,6066,0.061,0.001,0.059,0.062


In [10]:
print("Innovations on raw predicted states (Monte Carlo averages and rejection rates):")
_sort_summary(mc_reference["measurement_regressions_raw_summary"]).round(3)

Innovations on raw predicted states (Monte Carlo averages and rejection rates):


,measurement,predictor,coef,standardized_coef,std_error,t_stat,p_value,r2,mc_se_coef,mc_se_standardized_coef,mc_se_std_error,mc_se_t_stat,mc_se_p_value,mc_se_r2,n_replications,n_rejections,reject_rate,reject_rate_mc_se,reject_ci_low,reject_ci_high
2,OutGap,Pi,0.232,0.018,0.943,0.259,0.509,0.005,0.003,0.0,0.000,0.003,0.001,0.0,100000,4216,0.042,0.001,0.041,0.043
1,OutGap,x,-0.137,-0.036,0.241,-0.509,0.498,0.005,0.001,0.0,0.000,0.003,0.001,0.0,100000,4788,0.048,0.001,0.047,0.049
0,OutGap,r,-1.421,-0.041,2.481,-0.575,0.461,0.006,0.008,0.0,0.001,0.003,0.001,0.0,100000,7603,0.076,0.001,0.074,0.078
5,Infl,Pi,-0.382,-0.023,1.122,-0.322,0.486,0.005,0.004,0.0,0.000,0.003,0.001,0.0,100000,5985,0.060,0.001,0.058,0.061
4,Infl,x,-0.032,-0.005,0.288,-0.067,0.503,0.005,0.001,0.0,0.000,0.003,0.001,0.0,100000,4781,0.048,0.001,0.047,0.049
3,Infl,r,-0.066,-0.001,2.957,-0.009,0.505,0.005,0.009,0.0,0.001,0.003,0.001,0.0,100000,4561,0.046,0.001,0.044,0.047
8,Rate,Pi,0.038,0.011,0.225,0.154,0.497,0.005,0.001,0.0,0.000,0.003,0.001,0.0,100000,5211,0.052,0.001,0.051,0.054
7,Rate,x,0.032,0.036,0.057,0.515,0.461,0.006,0.000,0.0,0.000,0.003,0.001,0.0,100000,7993,0.080,0.001,0.078,0.082
6,Rate,r,-0.164,-0.016,0.591,-0.225,0.493,0.005,0.002,0.0,0.000,0.003,0.001,0.0,100000,5494,0.055,0.001,0.054,0.056


In [22]:
print("Innovation decomposition orthogonal summary:")
_sort_summary(mc_reference["innovation_decomposition_orthogonalized_summary"])

Innovation decomposition orthogonal summary:


,measurement,predictor,beta_measurement_error,beta_state_prediction_error,beta_total_innovation,beta_component_sum,beta_component_gap,abs_beta_component_gap,reconstruction_max_abs_error,mc_se_beta_measurement_error,mc_se_beta_state_prediction_error,mc_se_beta_total_innovation,mc_se_beta_component_sum,mc_se_beta_component_gap,mc_se_abs_beta_component_gap,mc_se_reconstruction_max_abs_error
0,OutGap,Pi,1.761303,-0.779137,0.982166,0.982166,-1.987567e-18,3.258722e-16,1.804376e-15,0.003008,0.002106,0.004286,0.004286,1.365185e-18,8.954502e-19,8.185675e-19
1,OutGap,x,-0.013789,-0.372118,-0.385907,-0.385907,-2.222035e-19,8.223633e-17,1.804376e-15,0.000779,0.000640,0.001116,0.001116,3.552315e-19,2.419939e-19,8.185675e-19
2,OutGap,r,-0.248766,-1.503490,-1.752257,-1.752257,2.417482e-18,5.885466e-16,1.804376e-15,0.006133,0.004993,0.008703,0.008703,2.505119e-18,1.676834e-18,8.185675e-19
3,Infl,Pi,-0.122614,-0.656700,-0.779315,-0.779315,-1.766710e-17,3.860494e-16,1.804376e-15,0.002305,0.004558,0.005057,0.005057,1.600801e-18,1.036980e-18,8.185675e-19
4,Infl,x,0.022236,0.045650,0.067886,0.067886,-5.375725e-19,9.726143e-17,1.804376e-15,0.000594,0.001196,0.001310,0.001310,4.039919e-19,2.619418e-19,8.185675e-19
5,Infl,r,-0.186423,-0.432528,-0.618950,-0.618950,-3.715888e-18,7.582112e-16,1.804376e-15,0.004671,0.009375,0.010249,0.010249,3.161122e-18,2.060080e-18,8.185675e-19
6,Rate,Pi,-0.002921,-0.078005,-0.080926,-0.080926,4.854339e-20,1.315866e-16,1.804376e-15,0.000498,0.000937,0.001022,0.001022,5.283348e-19,3.255541e-19,8.185675e-19
7,Rate,x,-0.000293,0.048777,0.048484,0.048484,1.041522e-19,3.402211e-17,1.804376e-15,0.000128,0.000262,0.000268,0.000268,1.374976e-19,8.561861e-20,8.185675e-19
8,Rate,r,-0.049811,-0.085877,-0.135688,-0.135688,1.570757e-18,2.681280e-16,1.804376e-15,0.001000,0.002049,0.002096,0.002096,1.079455e-18,6.680670e-19,8.185675e-19


In [23]:
print("Innovation decomposition raw summary:")
_sort_summary(mc_reference["innovation_decomposition_raw_summary"])

Innovation decomposition raw summary:


,measurement,predictor,beta_measurement_error,beta_state_prediction_error,beta_total_innovation,beta_component_sum,beta_component_gap,abs_beta_component_gap,reconstruction_max_abs_error,mc_se_beta_measurement_error,mc_se_beta_state_prediction_error,mc_se_beta_total_innovation,mc_se_beta_component_sum,mc_se_beta_component_gap,mc_se_abs_beta_component_gap,mc_se_reconstruction_max_abs_error
0,OutGap,Pi,1.816534,-1.584590,0.231944,0.231944,9.872321e-19,2.769296e-16,1.804376e-15,0.002113,0.001170,0.002778,0.002778,1.124104e-18,7.047770e-19,8.185675e-19
1,OutGap,x,0.319048,-0.456421,-0.137372,-0.137372,4.837880e-19,6.808735e-17,1.804376e-15,0.000513,0.000428,0.000688,0.000688,2.794128e-19,1.780857e-19,8.185675e-19
2,OutGap,r,-1.479585,0.058486,-1.421099,-1.421099,8.064993e-19,5.507728e-16,1.804376e-15,0.006557,0.004084,0.007537,0.007537,2.313604e-18,1.522903e-18,8.185675e-19
3,Infl,Pi,-0.020636,-0.361195,-0.381831,-0.381831,-1.788314e-17,2.707523e-16,1.804376e-15,0.001647,0.003185,0.003544,0.003544,1.119656e-18,7.237081e-19,8.185675e-19
4,Infl,x,0.003123,-0.035564,-0.032441,-0.032441,-3.607156e-18,6.910051e-17,1.804376e-15,0.000422,0.000836,0.000910,0.000910,2.853716e-19,1.838961e-19,8.185675e-19
5,Infl,r,-0.100604,0.034357,-0.066247,-0.066247,1.766733e-17,7.040075e-16,1.804376e-15,0.004339,0.008517,0.009247,0.009247,2.913060e-18,1.879554e-18,8.185675e-19
6,Rate,Pi,0.000688,0.037174,0.037862,0.037862,2.514045e-19,9.330863e-17,1.804376e-15,0.000355,0.000635,0.000711,0.000711,3.742366e-19,2.301908e-19,8.185675e-19
7,Rate,x,0.000254,0.031841,0.032096,0.032096,9.923608e-20,2.409818e-17,1.804376e-15,0.000091,0.000180,0.000187,0.000187,9.744348e-20,6.072933e-20,8.185675e-19
8,Rate,r,-0.032117,-0.131450,-0.163567,-0.163567,8.070386e-19,2.526085e-16,1.804376e-15,0.000934,0.001849,0.001908,0.001908,1.016028e-18,6.278546e-19,8.185675e-19


Monte Carlo summaries above aggregate `_MC_SAMPLES` independent draws. The plots and MCMC output below continue on a representative first draw.

In [11]:
parser_aug = ModelParser(_AUGMENTED_CONFIG)
config_aug, kalman_aug = parser_aug.get_all()
solver_aug = DSGESolver(config_aug, kalman_aug)
comp_aug = solver_aug.compile(
    n_state=3,
    n_exog=3,
)
priors = {
    _AUGMENTED_PARAM: make_prior(
        'normal',
        parameters={"mean": 0.0, "std": 4.0, "random_state": next_seed(_plot_seed_counter)},
        transform="identity",
    ),
}

with STDOUT_VOID():
    mc_aug = summarize_mle_augmentation_experiment(
        sol,
        solver_aug,
        comp_aug,
        sol_dgp,
        mc_reference,
        T=T,
        candidate_param=_AUGMENTED_PARAM,
        mc_samples=_MC_SAMPLES,
        alpha=_MC_ALPHA,
        summary_only=_MC_SUMMARY_ONLY,
    )

# estim = lambda: solver_aug.estimate_and_solve(
#     compiled=comp_aug,
#     method="mcmc",
#     n_draws=25_000,
#     burn_in=10_000,
#     thin=2,
#     posterior_point="mean",
#     proposal_scale=1.0,
#     y=obs,
#     priors=priors,
#     steady_state=[0.0, 0.0, 0.0, 0.0, 0.0],
#     random_state=next_seed(_plot_seed_counter),
#     **mc_reference["filter_kwargs"],
# )
# res_aug, sol_aug = estim()


## Diagnostics of the Augmented Model

### Marginal LR Test Conditional on $\theta_0$

In [12]:
print("Monte Carlo LR summary for the MLE-augmented model:")
rep_aug = mc_aug["representative"]
res_mle = rep_aug.res_mle
sol_mle = rep_aug.sol_mle
mle_aug_kf = rep_aug.kf_aug
std_innov_aug_mle = rep_aug.std_innov_aug
sim_aug_mle = rep_aug.sim_aug
mc_aug["lr_summary"].round(3)

Monte Carlo LR summary for the MLE-augmented model:


,estimated_coef,loglik_ref,loglik_aug,lr,p_value,mc_se_estimated_coef,mc_se_loglik_ref,mc_se_loglik_aug,mc_se_lr,mc_se_p_value,n_replications,n_rejections,reject_rate,reject_rate_mc_se,reject_ci_low,reject_ci_high
0,0.049,-1330.926,-1329.067,3.718,0.218,0.0,0.091,0.091,0.011,0.001,100000,38192,0.382,0.002,0.379,0.385


In [13]:
res_mle

OptimizationResult(kind='mle', x=array([0.05938168]), theta={'beta': np.float64(0.971), 'kappa': np.float64(0.58), 'tau_inv': np.float64(1.86), 'psi_pi': np.float64(2.19), 'psi_x': np.float64(0.3), 'rho_r': np.float64(0.84), 'rho_g': np.float64(0.83), 'rho_z': np.float64(0.85), 'pi_star': np.float64(3.43), 'r_star': np.float64(3.01), 'sig_r': np.float64(0.18), 'sig_g': np.float64(0.18), 'sig_z': np.float64(0.64), 'rho_gz': np.float64(0.36), 'meas_infl': np.float64(1e-06), 'meas_rate': np.float64(1e-06), 'meas_outgap': np.float64(1e-06), 'meas_rho_ir': np.float64(0.0), 'meas_rho_gi': np.float64(0.0), 'meas_rho_gr': np.float64(0.0), 'Pi_coef': np.float64(0.0), 'x_coef': np.float64(0.05938168362687965), 'r_coef': np.float64(0.0)}, success=True, message='CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH', fun=np.float64(1290.0894598385482), loglik=np.float64(-1290.0894598385482), logprior=np.float64(0.0), logpost=np.float64(-1290.0894598385482), nfev=12, nit=4, raw=  message: CONVERGENC

## Serial Autocorrelation Tests for the Augmented Model

In [14]:
print("Monte Carlo Ljung-Box summary for the MLE-augmented model:")
display(mc_aug["lb_summary"].round(3))

Monte Carlo Ljung-Box summary for the MLE-augmented model:


,measurement,lb_stat,p_value,mc_se_lb_stat,mc_se_p_value,n_replications,n_rejections,reject_rate,reject_rate_mc_se,reject_ci_low,reject_ci_high
0,OutGap,1.197,0.466,0.005,0.001,100000,7238,0.072,0.001,0.071,0.074
1,Infl,1.771,0.392,0.007,0.001,100000,14285,0.143,0.001,0.141,0.145
2,Rate,1.003,0.497,0.004,0.001,100000,4937,0.049,0.001,0.048,0.051


In [15]:
print("Reference moment-specification test summary:")
display(mc_reference["moment_specification_test_summary"].round(3))

print("Augmented moment-specification test summary:")
display(mc_aug["moment_specification_test_summary"].round(3))

print("Reference-minus-augmented moment distance comparison:")
display(mc_aug["moment_specification_comparison"].round(3))

Reference moment-specification test summary:


,test,distance,stat,p_value,mc_se_distance,mc_se_stat,mc_se_p_value,n_replications,n_rejections,reject_rate,reject_rate_mc_se,reject_ci_low,reject_ci_high,df,sample_size,bandwidth
0,mean_zero_hac,0.119,3.175,0.491,0.000,0.009,0.001,100000,6697,0.067,0.001,0.065,0.069,3.0,200,4
1,cov_identity,1.700,162.074,0.000,0.001,0.193,0.000,100000,100000,1.000,0.000,1.000,1.000,6.0,200,4


Augmented moment-specification test summary:


,test,distance,stat,p_value,mc_se_distance,mc_se_stat,mc_se_p_value,n_replications,n_rejections,reject_rate,reject_rate_mc_se,reject_ci_low,reject_ci_high,df,sample_size,bandwidth
0,mean_zero_hac,0.119,3.161,0.492,0.000,0.009,0.001,100000,6594,0.066,0.001,0.064,0.067,3.0,200,4
1,cov_identity,1.692,149.150,0.000,0.001,0.177,0.000,100000,100000,1.000,0.000,1.000,1.000,6.0,200,4


Reference-minus-augmented moment distance comparison:


""
